# Notebook 03 : évaluer la recherche du volet A

**Objet.** Mesurer la qualité de la recherche du volet A sur le jeu de questions de l'équipe (#44) et trancher
par la mesure les choix encore ouverts : critère du rapport J2, découpage servi, réglages, précision du modèle,
abstention. Ce notebook applique **à la lettre** le protocole `docs/etudes/evaluation/03_protocole.md`, écrit
avant toute mesure.

**Trois modes.**

- `developpement` : le notebook ne lit que le **lot de calibration** (8 questions). Il sert à vérifier chaque
  étape. Ses chiffres n'ont **aucune valeur de mesure**.
- `mesure` : le notebook lit le lot de test, une seule fois, et seulement si les empreintes du protocole figé et
  du jeu figé sont celles inscrites en étape 1. C'est ce verrou qui garantit qu'aucun choix n'est fait après
  avoir vu les résultats.
- `essai` : répète le chemin exact du mode mesure (analyse sur un lot, seuil sur l'autre), mais sur le seul lot de
  calibration, dont 3 questions du corpus et 2 hors corpus jouent le rôle du test. Il ne lit jamais le vrai lot
  de test ; ses chiffres ne mesurent rien.

**Plan.**

1. Configuration et verrou
2. Contrôle des modules
3. Entrées et empreintes
4. Règle de pertinence
5. Encodage des questions
6. Classements de tous les systèmes
7. Tableau des métriques
8. Analyse principale : le critère du J2
9. Analyses secondaires
10. Sigles
11. Abstention
12. Complément CDTN
13. Sauvegarde et bilan

Rappels utiles : le support *Comprendre l'encodeur multilingual-e5-small* et la fiche *Entraîner un encodeur de
recherche* (`docs/etudes/indexation/`), et le rapport de validation des métriques sur PIAF
(`docs/etudes/evaluation/`).

## Étape 1 : configuration et verrou

**Pourquoi.** Un résultat ne vaut que si la méthode a été fixée avant de le voir. Le notebook refuse donc de lire
le lot de test tant que le protocole et le jeu ne sont pas figés, et il vérifie leurs empreintes.

**Comment.** Les empreintes attendues sont inscrites ci-dessous au moment du gel ; tant qu'elles valent `None`,
seul le mode `developpement` est possible.

In [1]:
import os
# ni barres de progression ni avertissements réseau dans les sorties du notebook
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import hashlib
import json
import math
import re
import sys
import time
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# racine du dépôt : le premier dossier parent qui contient .git (fichier ou dossier)
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(RACINE / "scripts" / "evaluation"))
from metrics import (rank_by_score, evaluate_retrieval, paired_success_test, proportion_ci, holm,
                     random_expectations, sanity_check_manual_examples)
from pertinence import IndexPertinence, sanity_check as controle_pertinence
from lexical_baselines import BM25Retriever, TfidfRetriever, sanity_check as controle_lexical

MODE = os.environ.get("NB03_MODE", "developpement")
PROTOCOLE = RACINE / "docs" / "etudes" / "evaluation" / "03_protocole.md"
PROTOCOLE_SHA256 = None    # inscrit au gel du protocole
JEU = RACINE / "evaluation" / "questions" / "questions_annotees.csv"
JEU_SHA256 = None          # inscrit au gel du jeu (#44)
SORTIES = RACINE / "data" / "evaluation" / "notebook03"
SORTIES.mkdir(parents=True, exist_ok=True)

MODELE, REVISION, PREFIXE = "intfloat/multilingual-e5-small", "614241f622f53c4eeff9890bdc4f31cfecc418b3", "query: "
DECOUPAGES = ["m0", "m1", "m2", "m2_256", "m3"]
# index d'évaluation du notebook 02 -> table de passages dont il est issu (pertinence et métadonnées)
INDEX = {"m2": "m2", "m2_sans_entete": "m2", "m2_256": "m2_256", "m0": "m0", "m1": "m1", "m3": "m3"}
K_VALUES, PROFONDEUR, K_PRINCIPAL = (1, 3, 5, 10), 10, 5
MARGE = 0.15                 # marge de non-infériorité (D2)
GRAINE, TIRAGES = 20261007, 10000
AUCUN = "__aucun_resultat__" # complète une liste lexicale trop courte : jamais pertinent (D5)
DISCORDANCES_MIN = 10         # en dessous, intervalle d'Agresti et Min au lieu du Wald robuste (protocole, section 5)
SIGLES = {"CDI": "contrat à durée indéterminée", "CDD": "contrat à durée déterminée",
          "RTT": "réduction du temps de travail", "CSE": "comité social et économique",
          "SMIC": "salaire minimum interprofessionnel de croissance",
          "VAE": "validation des acquis de l'expérience", "CPF": "compte personnel de formation"}
MOTIF_SIGLES = re.compile(r"\b(" + "|".join(SIGLES) + r")\b")


def sha256_fichier(chemin: Path) -> str:
    return hashlib.sha256(Path(chemin).read_bytes()).hexdigest()


if MODE == "mesure":
    if PROTOCOLE_SHA256 is None or sha256_fichier(PROTOCOLE) != PROTOCOLE_SHA256:
        raise RuntimeError("Protocole non figé ou modifié : la mesure est refusée.")
    if JEU_SHA256 is None or not JEU.exists() or sha256_fichier(JEU) != JEU_SHA256:
        raise RuntimeError("Jeu de questions non figé ou modifié : la mesure est refusée.")
    LOTS, LOT_ANALYSE = ("test", "calibration"), "test"
elif MODE in ("developpement", "essai"):
    if not JEU.exists():
        # tant que #54 n'est pas fusionnée : copie de travail du jeu, hors du dépôt
        JEU = SORTIES / "questions_annotees.csv"
    LOTS = ("calibration",) if MODE == "developpement" else ("test", "calibration")
    LOT_ANALYSE = "calibration" if MODE == "developpement" else "test"
else:
    raise ValueError(f"Mode inconnu : {MODE}")

print(f"Mode : {MODE} | lots lus : {LOTS} | lot analysé : {LOT_ANALYSE}")
print(f"Protocole : {sha256_fichier(PROTOCOLE)[:16]}... | jeu : {JEU.relative_to(RACINE)} ({sha256_fichier(JEU)[:16]}...)")
if MODE != "mesure":
    print(f"ATTENTION : mode {MODE}. Les chiffres qui suivent portent sur le lot de calibration et ne mesurent rien.")

Mode : developpement | lots lus : ('calibration',) | lot analysé : calibration
Protocole : 863445c6d4c115b4... | jeu : data/evaluation/notebook03/questions_annotees.csv (807498144fec4672...)
ATTENTION : mode developpement. Les chiffres qui suivent portent sur le lot de calibration et ne mesurent rien.


**Conclusion.** Le mode et les lots lus sont fixés ; en mode développement, aucune ligne du lot de test ne sera
conservée. **Transition.** Avant d'utiliser les modules de calcul, on rejoue leurs contrôles.

## Étape 2 : contrôle des modules

**Pourquoi.** Les métriques, la règle de pertinence et les références lexicales viennent de trois modules. Chacun
porte des contrôles calculés à la main ; on les rejoue ici, dans l'environnement du notebook.

In [2]:
assert sanity_check_manual_examples() and controle_pertinence() and controle_lexical()
VERSIONS = {p: metadata.version(p) for p in ["numpy", "scipy", "scikit-learn", "torch", "transformers",
                                              "sentence-transformers", "rank-bm25", "snowballstemmer", "pandas"]}
print("Contrôles des trois modules : validés.")
print(VERSIONS)

Contrôles des trois modules : validés.
{'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'torch': '2.14.0', 'transformers': '5.17.0', 'sentence-transformers': '6.1.0', 'rank-bm25': '0.2.2', 'snowballstemmer': '3.1.1', 'pandas': '3.0.6'}


**Conclusion.** Les trois modules donnent les résultats calculés à la main. **Transition.** On charge ensuite les
données, en vérifiant chaque fichier.

## Étape 3 : entrées et empreintes

**Pourquoi.** Les passages et les index ont été produits par les notebooks 01 et 02. Un fichier modifié depuis, ou
un index désaligné de sa table, fausserait tout sans erreur visible.

**Comment.** On compare chaque fichier aux empreintes des manifestes des notebooks 01 et 02, on vérifie que l'ordre
des vecteurs suit celui des tables, et que l'index M2 est bien celui du site.

In [3]:
manifeste_01 = json.loads((RACINE / "data/decoupage/passages/manifeste.json").read_text(encoding="utf-8"))["methodes"]
variantes = json.loads((RACINE / "data/indexation/variantes/variantes.json").read_text(encoding="utf-8"))
entete_site = json.loads((RACINE / "data/indexation/site/index_m2.json").read_text(encoding="utf-8"))

PASSAGES, controles = {}, []
for m in DECOUPAGES:
    chemin = RACINE / manifeste_01[m]["fichier"]
    ok = sha256_fichier(chemin) == manifeste_01[m]["sha256_fichier"]
    PASSAGES[m] = pd.read_parquet(chemin)
    ok = ok and len(PASSAGES[m]) == manifeste_01[m]["passages"]
    controles.append({"fichier": chemin.name, "lignes": len(PASSAGES[m]), "conforme": ok})

VECTEURS = {}
for nom, table in INDEX.items():
    v = variantes[nom]
    chemin = RACINE / "data/indexation/variantes" / v["index"]
    ok = sha256_fichier(chemin) == v["sha256_index"]
    E = np.fromfile(chemin, dtype="<f4").reshape(v["nb_vecteurs"], 384)
    ids = json.loads((RACINE / "data/indexation/variantes" / v["identifiants"]).read_text(encoding="utf-8"))
    ok = ok and ids == PASSAGES[table].passage_id.tolist()
    ok = ok and np.allclose(np.linalg.norm(E, axis=1), 1.0, atol=1e-4)
    VECTEURS[nom] = (E, ids)
    controles.append({"fichier": v["index"], "lignes": v["nb_vecteurs"], "conforme": ok})
ok_site = entete_site["index"]["sha256"] == variantes["m2"]["sha256_index"]
controles.append({"fichier": "index_m2.f32 (site) = m2.f32", "lignes": entete_site["index"]["nb_vecteurs"], "conforme": ok_site})
controles = pd.DataFrame(controles)
display(controles)
assert controles.conforme.all(), "Une entrée ne correspond pas à son manifeste."

questions = pd.read_csv(JEU, dtype=str, keep_default_na=False)
if MODE == "essai":
    # le vrai lot de test est écarté d'abord ; puis 3 questions du corpus et 2 hors corpus de la calibration jouent le test
    questions = questions[questions.lot == "calibration"].reset_index(drop=True)
    for type_, nombre in (("dans_corpus", 3), ("hors_corpus", 2)):
        questions.loc[questions.index[questions.type == type_][:nombre], "lot"] = "test"
questions = questions[questions.lot.isin(LOTS)].reset_index(drop=True)   # le lot de test est écarté en développement
analyse = questions[questions.lot == LOT_ANALYSE].reset_index(drop=True)
dans = analyse[analyse.type == "dans_corpus"].reset_index(drop=True)
hors = analyse[analyse.type == "hors_corpus"].reset_index(drop=True)
FICHE = dans.url.str.split("#").str[0].str.rstrip("/").tolist()   # groupe de chaque question : sa fiche annotée
print(f"Lot analysé ({LOT_ANALYSE}) : {len(dans)} questions dans le corpus, {len(hors)} hors corpus, "
      f"{len(set(FICHE))} fiches distinctes")

,fichier,lignes,conforme
0,passages_m0.parquet,5284,True
1,passages_m1.parquet,5317,True
2,passages_m2.parquet,4240,True
3,passages_m2_256.parquet,7009,True
4,passages_m3.parquet,4229,True
5,m2.f32,4240,True
6,m2_sans_entete.f32,4240,True
7,m2_256.f32,7009,True
8,m0.f32,5284,True
9,m1.f32,5317,True


Lot analysé (calibration) : 5 questions dans le corpus, 3 hors corpus, 5 fiches distinctes


**Conclusion.** Les passages et les index sont ceux des notebooks 01 et 02, alignés, et l'index M2 est celui du
site. **Transition.** Il faut maintenant dire, pour chaque question, quels passages y répondent.

## Étape 4 : règle de pertinence (protocole, section 3)

**Rappel.** Un passage est pertinent s'il partage avec l'extrait annoté un fragment contigu qui couvre au moins
la moitié de l'extrait, ou au moins la moitié du passage avec un minimum de 80 caractères, après normalisation
(lettres et chiffres seuls, en minuscules), avec et sans retrait des marques de liste numérotées. La seconde
condition ne joue que pour les extraits longs, comme une réponse en liste, qu'aucun petit passage ne pourrait
contenir à moitié. Chaque découpage est jugé avec sa propre application de la règle : exiger l'extrait entier
avantagerait M2, dont les passages suivent les sections.

**Comment.** `IndexPertinence` présélectionne les passages par des fragments d'ancrage, puis vérifie la règle ; sa
présélection est exacte, et contrôlée contre une recherche exhaustive.

In [4]:
PERTINENTS, FICHES_PERTINENTES, lignes = {}, {}, []
for m in DECOUPAGES:
    P = PASSAGES[m]
    index_p = IndexPertinence(P.texte.tolist())
    rel = [index_p.pertinents(e) for e in dans.extrait]
    PERTINENTS[m] = [set(P.passage_id.iloc[r]) for r in rel]
    FICHES_PERTINENTES[m] = [set(P.url.iloc[r].str.rstrip("/")) for r in rel]
    n = [len(r) for r in rel]
    lignes.append({"découpage": m, "questions": len(n), "sans passage pertinent": sum(x == 0 for x in n),
                   "un seul": sum(x == 1 for x in n), "deux ou plus": sum(x >= 2 for x in n),
                   "médiane": float(np.median(n)) if n else float("nan")})
PERTINENTS["m2_sans_entete"], FICHES_PERTINENTES["m2_sans_entete"] = PERTINENTS["m2"], FICHES_PERTINENTES["m2"]
display(pd.DataFrame(lignes))
SANS_PERTINENT = {m: [dans.id[i] for i, r in enumerate(PERTINENTS[m]) if not r] for m in DECOUPAGES}
print("Questions sans passage pertinent, comptées en échec pour ce découpage :", {m: v for m, v in SANS_PERTINENT.items() if v})

,découpage,questions,sans passage pertinent,un seul,deux ou plus,médiane
0,m0,5,0,4,1,1.0
1,m1,5,0,4,1,1.0
2,m2,5,0,5,0,1.0
3,m2_256,5,0,5,0,1.0
4,m3,5,0,5,0,1.0


Questions sans passage pertinent, comptées en échec pour ce découpage : {}


**Conclusion.** Chaque question a son ensemble de passages pertinents, découpage par découpage ; une question sans
passage pertinent compterait comme un échec, et elle est signalée. **Transition.** Il reste à transformer les
questions en vecteurs.

## Étape 5 : encodage des questions

**Pourquoi.** Le site encode la question dans le navigateur, en q8 ; c'est la configuration « E5 servi » de
l'analyse principale. La référence fp32 se calcule en Python.

**Comment.**

- **fp32** : `sentence-transformers` à la révision figée, préfixe `query: `, moyenne des tokens et normalisation ;
  mis en cache sous une clé qui dépend des textes, de la révision et des versions.
- **q8** : le notebook écrit `questions_navigateur.json` ; la page `scripts/evaluation/encodage_navigateur.html`,
  servie sur 127.0.0.1, encode les questions avec la configuration de l'en-tête du site, et son résultat est
  enregistré dans `vecteurs_navigateur.json`. Le notebook contrôle l'empreinte, l'ordre et la fidélité au fp32.
- Les questions qui contiennent un sigle sont aussi encodées dans une version étendue (étape 10).

In [5]:
def etendre(texte: str) -> str:
    """Chaque sigle de la liste du protocole est suivi de son développement entre parenthèses."""
    return MOTIF_SIGLES.sub(lambda m: f"{m.group(1)} ({SIGLES[m.group(1)]})", texte)


# toutes les questions lues : en mode mesure, le seuil de secours de l'étape 11 se calcule sur le lot de calibration
A_ENCODER = [{"id": r.id, "texte": r.question} for r in questions.itertuples()]
A_ENCODER += [{"id": r.id + "+sigles", "texte": etendre(r.question)} for r in questions.itertuples()
              if MOTIF_SIGLES.search(r.question)]
TEXTES = [x["texte"] for x in A_ENCODER]
POSITION = {x["id"]: i for i, x in enumerate(A_ENCODER)}

cle = hashlib.sha256(json.dumps({"textes": TEXTES, "revision": REVISION, "prefixe": PREFIXE, "versions": VERSIONS},
                                ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]
cache = SORTIES / f"questions_fp32_{cle}.npy"
if cache.exists():
    U_FP32 = np.load(cache)
else:
    from huggingface_hub.utils import logging as journal_hub
    from sentence_transformers import SentenceTransformer
    journal_hub.set_verbosity_error()
    modele = SentenceTransformer(MODELE, revision=REVISION, device="cpu")
    U_FP32 = modele.encode([PREFIXE + t for t in TEXTES], normalize_embeddings=True,
                           convert_to_numpy=True).astype(np.float32)
    np.save(cache, U_FP32)
print(f"fp32 : {U_FP32.shape[0]} textes encodés (clé {cle})")

# questions pour la page du navigateur, contrôlées par l'empreinte de leur texte JSON
lot_json = json.dumps(A_ENCODER, ensure_ascii=False, separators=(",", ":"))
(SORTIES / "questions_navigateur.json").write_text(
    json.dumps({"prefixe": PREFIXE, "empreinte": hashlib.sha256(lot_json.encode("utf-8")).hexdigest(),
                "questions": A_ENCODER}, ensure_ascii=False), encoding="utf-8")
U_Q8, FIDELITE = None, None
fichier_q8 = SORTIES / "vecteurs_navigateur.json"
if fichier_q8.exists():
    r = json.loads(fichier_q8.read_text(encoding="utf-8"))
    attendu = {"depot": entete_site["modele"]["depot"], "revision": entete_site["modele"]["revision"],
               "dtype": entete_site["modele"]["dtype"], "prefixe": PREFIXE,
               "empreinte_questions": hashlib.sha256(lot_json.encode("utf-8")).hexdigest(),
               "identifiants": [x["id"] for x in A_ENCODER]}
    if all(r.get(k) == v for k, v in attendu.items()):
        U_Q8 = np.array(r["vecteurs"], dtype=np.float32)
        cos = np.sum(U_Q8 * U_FP32, axis=1)
        FIDELITE = {"cosinus_min": float(cos.min()), "cosinus_median": float(np.median(cos)),
                    "navigateur": r["navigateur"], "transformers_js": r["transformers_js"]}
        assert U_Q8.shape == U_FP32.shape and np.allclose(np.linalg.norm(U_Q8, axis=1), 1, atol=1e-3)
        print(f"q8 : {len(U_Q8)} vecteurs du navigateur, cosinus avec le fp32 : minimum {cos.min():.4f}, médiane {np.median(cos):.4f}")
    else:
        print("q8 : vecteurs_navigateur.json ne correspond pas aux questions écrites ; page à relancer.")
if U_Q8 is None:
    message = "q8 : vecteurs du navigateur absents ou périmés ; ouvrir encodage_navigateur.html, puis réexécuter."
    if MODE == "mesure":
        raise RuntimeError(message)
    print(message, "E5 servi est omis en mode développement.")

fp32 : 8 textes encodés (clé be08b16bfaae9b3f)
q8 : 8 vecteurs du navigateur, cosinus avec le fp32 : minimum 0.9928, médiane 0.9967


**Conclusion.** Les questions ont leurs vecteurs fp32, et leurs vecteurs q8 si la page du navigateur a été
exécutée sur la liste en cours. **Transition.** On peut classer les passages pour chaque système.

## Étape 6 : classements de tous les systèmes (protocole, section 4)

**Comment.**

- **Encodeur** : produit scalaire avec chaque index, puis `rank_by_score`, les ex aequo dans l'ordre de l'index.
- **Références lexicales** : `lexical_baselines.py`, sur l'en-tête et le texte des passages M2 ; seuls les passages
  de score positif sont classés, et une liste trop courte est complétée par un marqueur jamais pertinent (D5).
- **Contrôle indépendant** : pour chaque question, le rang du premier pertinent lu dans la liste doit tomber dans
  l'intervalle des ex aequo recalculé sans aucun tri, directement depuis la matrice des scores.

In [6]:
def classer(S: np.ndarray, ids: list, positifs: bool) -> list:
    listes = []
    for ligne in S:
        ordre = rank_by_score(ligne)
        if positifs:
            ordre = ordre[ligne[ordre] > 0]
        liste = [ids[j] for j in ordre[:PROFONDEUR]]
        listes.append(liste + [AUCUN] * (PROFONDEUR - len(liste)))
    return listes


def controle_rangs(S: np.ndarray, ids: list, listes: list, pertinents: list, positifs: bool) -> None:
    position = {p: j for j, p in enumerate(ids)}
    for ligne, liste, rel in zip(S, listes, pertinents):
        cols = [position[p] for p in rel]
        if not cols:
            continue
        cible = ligne[cols].max()
        bas, haut = 1 + int((ligne > cible).sum()), int((ligne >= cible).sum())
        rang = next((i + 1 for i, p in enumerate(liste) if p in rel), math.inf)
        valide = (positifs and cible <= 0) or (bas <= rang <= haut) or (rang == math.inf and haut > PROFONDEUR)
        assert valide, "rang incohérent avec la matrice des scores"


SYSTEMES = {}   # nom -> dict(table, listes)
lignes_q = [POSITION[i] for i in dans.id]
for nom in INDEX:
    E, ids = VECTEURS[nom]
    S = U_FP32[lignes_q] @ E.T
    SYSTEMES[f"E5 fp32, {nom}"] = {"table": INDEX[nom], "pertinence": nom, "S": S, "ids": ids,
                                   "listes": classer(S, ids, False), "positifs": False}
if U_Q8 is not None:
    E, ids = VECTEURS["m2"]
    S = U_Q8[lignes_q] @ E.T
    SYSTEMES["E5 servi"] = {"table": "m2", "pertinence": "m2", "S": S, "ids": ids,
                            "listes": classer(S, ids, False), "positifs": False}

P2 = PASSAGES["m2"]
textes_m2 = (P2.entete + "\n" + P2.texte).tolist()
assert (P2.entree_encodeur == PREFIXE.replace("query", "passage") + pd.Series(textes_m2)).all()
LEXICAL = {"V0": (None, None, 1, 1.0), "V1": ("snowball", None, 1, 1.0), "V2": ("snowball_no_negation", None, 1, 1.0),
           "V3": ("snowball_no_negation", "french", 1, 1.0), "V4": ("snowball_no_negation", "french", 2, 0.85)}
for code, (sw, st, mn, mx) in LEXICAL.items():
    for famille, classe in (("BM25", BM25Retriever), ("TF-IDF", TfidfRetriever)):
        moteur = classe(stopwords_variant=sw, stemming=st, min_df=mn, max_df=mx).fit(textes_m2, doc_ids=P2.passage_id.tolist())
        S = moteur.score_matrix(dans.question.tolist())
        nom = "BM25 de référence (V3)" if (famille, code) == ("BM25", "V3") else f"{famille} {code}"
        SYSTEMES[nom] = {"table": "m2", "pertinence": "m2", "S": S, "ids": P2.passage_id.tolist(),
                         "listes": classer(S, P2.passage_id.tolist(), True), "positifs": True}

for nom, s in SYSTEMES.items():
    controle_rangs(s["S"], s["ids"], s["listes"], PERTINENTS[s["pertinence"]], s["positifs"])
print(f"{len(SYSTEMES)} systèmes classés ; rangs contrôlés contre les matrices de scores.")

17 systèmes classés ; rangs contrôlés contre les matrices de scores.


**Conclusion.** Chaque système a son classement des passages pour chaque question, contrôlé sans tri.
**Transition.** On en tire les métriques.

## Étape 7 : tableau des métriques (protocole, section 5)

**Rappel.** Une question est réussie à k si au moins un passage pertinent figure parmi les k premiers ;
MRR@10 moyenne 1 / rang du premier pertinent, compté 0 au-delà de 10. Les intervalles tiennent compte du
regroupement des questions par fiche : Wilson calculé sur l'effectif corrigé de l'effet de plan, qui reste dans
[0, 1]. Le tirage aléatoire est donné par son espérance exacte.

In [7]:
def succes(nom: str, k: int) -> np.ndarray:
    s = SYSTEMES[nom]
    return np.array([any(p in rel for p in liste[:k]) for liste, rel in zip(s["listes"], PERTINENTS[s["pertinence"]])])


def succes_fiche(nom: str, k: int) -> np.ndarray:
    s = SYSTEMES[nom]
    url = dict(zip(PASSAGES[s["table"]].passage_id, PASSAGES[s["table"]].url.str.rstrip("/")))
    return np.array([any(url.get(p) in fiches for p in liste[:k])
                     for liste, fiches in zip(s["listes"], FICHES_PERTINENTES[s["pertinence"]])])


lignes = []
for nom, s in SYSTEMES.items():
    rel = PERTINENTS[s["pertinence"]]
    garder = [i for i, r in enumerate(rel) if r]           # evaluate_retrieval refuse un ensemble vide
    m = evaluate_retrieval([s["listes"][i] for i in garder], [rel[i] for i in garder], K_VALUES) if garder else {}
    # une question sans passage pertinent compte comme un échec (protocole, section 3)
    facteur = len(garder) / len(rel)
    bas, haut = proportion_ci(succes(nom, K_PRINCIPAL), groups=FICHE, method="wilson")
    assert (succes_fiche(nom, K_PRINCIPAL) >= succes(nom, K_PRINCIPAL)).all(), "succès fiche inférieur au succès passage"
    lignes.append({"système": nom, **{f"R@{k}": facteur * m.get(f"recall@{k}", 0.0) for k in K_VALUES},
                   "MRR@10": facteur * m.get("mrr", 0.0), "R@5 IC bas": bas, "R@5 IC haut": haut,
                   "R@5 fiche": succes_fiche(nom, K_PRINCIPAL).mean()})
aleatoire = [random_expectations(len(r), len(PASSAGES["m2"]), K_VALUES, PROFONDEUR) for r in PERTINENTS["m2"]]
lignes.append({"système": "Aléatoire (espérance, M2)", **{f"R@{k}": np.mean([a[f"recall@{k}"] for a in aleatoire]) for k in K_VALUES},
               "MRR@10": np.mean([a["mrr"] for a in aleatoire])})
METRIQUES = pd.DataFrame(lignes).set_index("système")
for nom in SYSTEMES:   # recoupement : la part de succès à 5 redonne Recall@5
    assert np.isclose(METRIQUES.loc[nom, "R@5"], succes(nom, 5).mean())
display(METRIQUES.round(3))

,R@1,R@3,R@5,R@10,MRR@10,R@5 IC bas,R@5 IC haut,R@5 fiche
système,,,,,,,,
"E5 fp32, m2",0.4,0.400,0.600,0.800,0.460,0.231,0.882,1.0
"E5 fp32, m2_sans_entete",0.2,0.400,0.400,0.600,0.325,0.118,0.769,1.0
"E5 fp32, m2_256",0.2,0.400,0.400,0.600,0.320,0.118,0.769,1.0
"E5 fp32, m0",0.4,0.400,0.600,0.800,0.473,0.231,0.882,0.8
"E5 fp32, m1",0.2,0.200,0.200,0.400,0.233,0.036,0.624,0.8
"E5 fp32, m3",0.2,0.400,0.400,0.800,0.322,0.118,0.769,0.8
E5 servi,0.4,0.400,0.600,0.600,0.440,0.231,0.882,1.0
BM25 V0,0.2,0.400,0.400,0.400,0.300,0.118,0.769,0.8
TF-IDF V0,0.2,0.400,0.400,0.600,0.295,0.118,0.769,0.8


**Conclusion.** Le tableau donne, pour chaque système, Recall@k, MRR@10, l'intervalle de Recall@5 et le succès
au niveau de la fiche. **Transition.** L'analyse principale compare deux de ces lignes.

## Étape 8 : analyse principale, le critère du J2 (protocole, section 6)

**Question.** E5 servi est-il non inférieur à BM25 de référence, en Recall@5 ? L'écart d et son intervalle à
95 %, robuste au regroupement par fiche, décident : critère satisfait si la borne basse dépasse -15 points ;
E5 servi meilleur si elle dépasse aussi 0 ; sinon, critère non démontré.

In [8]:
def intervalle(t: dict) -> tuple:
    """Intervalle de l'écart selon le protocole : Wald robuste aux groupes, ou Agresti et Min si les paires
    discordantes sont rares (un Wald dégénère alors, jusqu'à une largeur nulle)."""
    if t["discordant"] < DISCORDANCES_MIN:
        return t["ci_difference_agresti_min"], "Agresti et Min"
    return t["ci_difference"], "Wald robuste"


PRINCIPALE = None
if "E5 servi" in SYSTEMES:
    t = paired_success_test(succes("E5 servi", K_PRINCIPAL), succes("BM25 de référence (V3)", K_PRINCIPAL), groups=FICHE)
    (bas, haut), methode = intervalle(t)
    verdict = ("satisfait, et E5 servi meilleur" if bas > 0 else "satisfait" if bas > -MARGE else "non démontré")
    PRINCIPALE = {**t, "marge": MARGE, "verdict": verdict, "intervalle_retenu": [bas, haut], "methode_intervalle": methode}
    print(f"Écart d = {100 * t['difference']:+.1f} points, IC 95 % [{100 * bas:+.1f} ; {100 * haut:+.1f}] ({methode}, "
          f"{t['discordant']} paires discordantes) ; p exact {t['p_value']:.3g} ; p groupé {t['p_value_groups']:.3g}")
    print(f"Table de contingence : {t['contingency_table']}")
    print(f"Critère du J2 : {verdict}" + ("  (DÉMONSTRATION sur le lot de calibration, sans valeur)" if MODE != "mesure" else ""))
else:
    print("Analyse principale impossible sans les vecteurs q8 du navigateur.")

Écart d = +20.0 points, IC 95 % [-23.9 ; +52.4] (Agresti et Min, 1 paires discordantes) ; p exact 1 ; p groupé 0.371
Table de contingence : {'n_00': 2, 'n_10_A_only': 1, 'n_01_B_only': 0, 'n_11': 2}
Critère du J2 : non démontré  (DÉMONSTRATION sur le lot de calibration, sans valeur)


**Conclusion.** Le verdict suit la règle fixée à l'avance, sans autre lecture. **Transition.** Les analyses
secondaires portent sur les réglages.

## Étape 9 : analyses secondaires (protocole, section 7)

**Familles.** F1 : M0, M1, M3 contre M2 ; F2 : M2 à 256 tokens et M2 sans en-tête contre M2 ; F3 : E5 servi
contre E5 fp32 sur M2. Chaque famille est corrigée par Holm ; la valeur p est celle du test d'Obuchowski, les
questions étant groupées par fiche. Deux découpages ont des passages pertinents différents : chaque succès est
calculé avec ceux de son propre découpage.

In [9]:
familles = {"F1 découpages": [("E5 fp32, m0", "E5 fp32, m2"), ("E5 fp32, m1", "E5 fp32, m2"), ("E5 fp32, m3", "E5 fp32, m2")],
            "F2 réglages de M2": [("E5 fp32, m2_256", "E5 fp32, m2"), ("E5 fp32, m2_sans_entete", "E5 fp32, m2")],
            "F3 précision servie": [("E5 servi", "E5 fp32, m2")]}
lignes = []
for famille, paires in familles.items():
    paires = [p for p in paires if p[0] in SYSTEMES and p[1] in SYSTEMES]
    tests = [paired_success_test(succes(a, K_PRINCIPAL), succes(b, K_PRINCIPAL), groups=FICHE) for a, b in paires]
    ajustees = holm([t["p_value_groups"] for t in tests])
    for (a, b), t, p_holm in zip(paires, tests, ajustees):
        (bas, haut), methode = intervalle(t)
        lignes.append({"famille": famille, "comparaison": f"{a} contre {b}", "écart": t["difference"],
                       "discordantes": t["discordant"], "IC bas": bas, "IC haut": haut, "intervalle": methode,
                       "p groupé": t["p_value_groups"], "p Holm": p_holm, "significatif": p_holm < 0.05})
SECONDAIRES = pd.DataFrame(lignes)
display(SECONDAIRES.round(3))

,famille,comparaison,écart,discordantes,IC bas,IC haut,intervalle,p groupé,p Holm,significatif
0,F1 découpages,"E5 fp32, m0 contre E5 fp32, m2",0.0,2,-0.485,0.485,Agresti et Min,1.000,1.000,False
1,F1 découpages,"E5 fp32, m1 contre E5 fp32, m2",-0.4,2,-0.722,0.151,Agresti et Min,0.206,0.618,False
2,F1 découpages,"E5 fp32, m3 contre E5 fp32, m2",-0.2,1,-0.524,0.239,Agresti et Min,0.371,0.742,False
3,F2 réglages de M2,"E5 fp32, m2_256 contre E5 fp32, m2",-0.2,1,-0.524,0.239,Agresti et Min,0.371,0.742,False
4,F2 réglages de M2,"E5 fp32, m2_sans_entete contre E5 fp32, m2",-0.2,1,-0.524,0.239,Agresti et Min,0.371,0.742,False
5,F3 précision servie,"E5 servi contre E5 fp32, m2",0.0,0,-0.280,0.280,Agresti et Min,1.000,1.000,False


**Conclusion.** Chaque réglage n'est remis en cause que par un écart significatif après correction. **Transition.**
Les deux dernières analyses, descriptives, portent sur les sigles et sur l'abstention.

## Étape 10 : sigles (protocole, section 8.1)

**Comment.** Pour chaque question du corpus qui contient un sigle de la liste, le rang du premier passage
pertinent avant et après extension du sigle (« CDI (contrat à durée indéterminée) »), pour E5 servi, E5 fp32 sur
M2 et BM25 de référence. Analyse descriptive : trop peu de questions pour un test.

In [10]:
def rang_premier(liste: list, rel: set) -> float:
    return next((i + 1 for i, p in enumerate(liste) if p in rel), math.inf)


avec_sigle = [i for i, q in enumerate(dans.question) if MOTIF_SIGLES.search(q)]
lignes = []
bm25_ref = BM25Retriever(stopwords_variant="snowball_no_negation", stemming="french").fit(textes_m2, doc_ids=P2.passage_id.tolist())
for i in avec_sigle:
    qid, rel = dans.id[i], PERTINENTS["m2"][i]
    ligne = {"question": qid}
    for nom, U in (("E5 fp32", U_FP32), ("E5 servi", U_Q8)):
        if U is None:
            continue
        E, ids = VECTEURS["m2"]
        for etat, cle_q in (("avant", qid), ("après", qid + "+sigles")):
            ligne[f"{nom} {etat}"] = rang_premier(classer(U[[POSITION[cle_q]]] @ E.T, ids, False)[0], rel)
    for etat, texte in (("avant", dans.question[i]), ("après", etendre(dans.question[i]))):
        ligne[f"BM25 {etat}"] = rang_premier(classer(bm25_ref.score_matrix([texte]), P2.passage_id.tolist(), True)[0], rel)
    lignes.append(ligne)
SIGLES_RESULTATS = pd.DataFrame(lignes)
print(f"{len(avec_sigle)} question(s) avec sigle dans le lot analysé.")
if len(SIGLES_RESULTATS):
    display(SIGLES_RESULTATS)

0 question(s) avec sigle dans le lot analysé.


**Conclusion.** Le tableau montre, question par question, si l'extension aide ; il ne prouve rien d'une seule
question. **Transition.** Reste la question des questions hors corpus.

## Étape 11 : abstention (protocole, section 8.2)

**Comment.** Le score d'une question est son plus grand cosinus avec les passages M2, pour E5 servi. L'AUC mesure,
sans seuil, la séparation entre questions du corpus et questions hors corpus, avec un intervalle par
rééchantillonnage stratifié. Le seuil, exploratoire, vient du CDTN (#56) ou, à défaut, du lot de calibration.

In [11]:
ABSTENTION = None
if U_Q8 is not None:
    E, _ = VECTEURS["m2"]
    score_max = lambda tab: np.array([float((U_Q8[POSITION[i]] @ E.T).max()) for i in tab.id])
    s_dans, s_hors = score_max(dans), score_max(hors)
    if len(s_dans) and len(s_hors):
        y = np.r_[np.ones(len(s_dans)), np.zeros(len(s_hors))]
        auc = roc_auc_score(y, np.r_[s_dans, s_hors])
        rng = np.random.default_rng(GRAINE)
        boot = [roc_auc_score(y, np.r_[rng.choice(s_dans, len(s_dans)), rng.choice(s_hors, len(s_hors))]) for _ in range(TIRAGES)]
        fichier_seuil = SORTIES / "seuil_cdtn.json"   # écrit par l'analyse du CDTN (#56)
        if fichier_seuil.exists():
            seuil, origine = float(json.loads(fichier_seuil.read_text(encoding="utf-8"))["seuil"]), "CDTN (#56)"
        else:
            cal = questions[(questions.lot == "calibration") & (questions.type == "dans_corpus")]
            seuil = float(min((U_Q8[POSITION[i]] @ E.T).max() for i in cal.id))
            origine = "lot de calibration, le CDTN (#56) n'étant pas prêt"
            if LOT_ANALYSE == "calibration":
                origine += " ; calcul circulaire en développement : seuil et mesure portent sur les mêmes questions"
        rejetes, conserves = int((s_hors < seuil).sum()), int((s_dans >= seuil).sum())
        ABSTENTION = {"auc": auc, "auc_ic": [float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))],
                      "seuil": seuil, "origine_seuil": origine,
                      "hors_rejetees": [rejetes, len(s_hors), *proportion_ci([1] * rejetes + [0] * (len(s_hors) - rejetes))],
                      "dans_conservees": [conserves, len(s_dans), *proportion_ci([1] * conserves + [0] * (len(s_dans) - conserves))]}
        print(f"AUC {auc:.3f}, IC 95 % [{ABSTENTION['auc_ic'][0]:.3f} ; {ABSTENTION['auc_ic'][1]:.3f}] ; "
              f"seuil {seuil:.4f} ({ABSTENTION['origine_seuil']})")
        print(f"Hors corpus rejetées : {rejetes}/{len(s_hors)} ; dans le corpus conservées : {conserves}/{len(s_dans)}")
else:
    print("Abstention impossible sans les vecteurs q8 du navigateur.")

AUC 0.800, IC 95 % [0.333 ; 1.000] ; seuil 0.8660 (lot de calibration, le CDTN (#56) n'étant pas prêt ; calcul circulaire en développement : seuil et mesure portent sur les mêmes questions)
Hors corpus rejetées : 2/3 ; dans le corpus conservées : 5/5


**Conclusion.** L'AUC dit si un seuil peut séparer les deux familles de questions ; le seuil, lui, reste à
confirmer sur le CDTN. **Transition.** Le complément CDTN est analysé à part.

## Étape 12 : complément CDTN (protocole, section 8.3)

Analyse secondaire séparée, en attente de l'issue #56 : appariement figé des requêtes aux passages M2, variantes
relues, pertinence jugée à la section, variantes groupées par requête. Elle ne sera jamais fusionnée avec les
résultats du jeu de l'équipe.

## Étape 13 : sauvegarde et bilan

**Comment.** Les rangs par question et par système, les métriques, les tests et un manifeste (mode, empreintes,
versions) sont écrits dans `data/evaluation/notebook03/`, hors du dépôt.

In [12]:
rangs = pd.DataFrame({"question": dans.id, "fiche": FICHE})
for nom, s in SYSTEMES.items():
    rangs[nom] = [rang_premier(liste, rel) for liste, rel in zip(s["listes"], PERTINENTS[s["pertinence"]])]
rangs.to_csv(SORTIES / f"rangs_{MODE}.csv", index=False)
resultats = {"mode": MODE, "lot": LOT_ANALYSE, "questions_dans": len(dans), "questions_hors": len(hors),
             "metriques": METRIQUES.reset_index().to_dict(orient="records"), "principale": PRINCIPALE,
             "secondaires": SECONDAIRES.to_dict(orient="records"), "sigles": SIGLES_RESULTATS.to_dict(orient="records"),
             "abstention": ABSTENTION, "fidelite_q8": FIDELITE}
(SORTIES / f"resultats_{MODE}.json").write_text(json.dumps(resultats, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
manifeste = {"mode": MODE, "protocole_sha256": sha256_fichier(PROTOCOLE), "jeu": str(JEU.relative_to(RACINE)),
             "jeu_sha256": sha256_fichier(JEU), "modele": MODELE, "revision": REVISION, "versions": VERSIONS,
             "site": entete_site["modele"], "execute_le": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
(SORTIES / f"manifeste_{MODE}.json").write_text(json.dumps(manifeste, ensure_ascii=False, indent=1), encoding="utf-8")
print("Sorties écrites :", sorted(p.name for p in SORTIES.glob(f"*_{MODE}.*")))

Sorties écrites : ['manifeste_developpement.json', 'rangs_developpement.csv', 'resultats_developpement.json']


**Bilan.** En mode développement, ce notebook vérifie que chaque étape s'exécute et se contrôle sur le lot de
calibration ; ses chiffres ne mesurent rien. La mesure se fera en mode `mesure`, une seule fois, après le gel du
jeu (#44) et la validation du protocole, dont les empreintes seront inscrites en étape 1.